# 602 offline LSTM capacity sweep vs offline stride
This notebook performs corrected chronological stateful-TBPTT training and causal offline inference on one Colab A100. Hidden/cell values cross chunk boundaries; only the computation graph is detached. It generates independent replay lists for several LSTM widths. ChampSim is not run here, and no historical all-trace pipeline is used.

Important: the first width that beats stride on the evaluation window is an exploratory saturation point. Confirm that selected width on a new held-out window before making a final model-selection claim.

In [ ]:
import os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > A100 GPU'
torch.set_float32_matmul_precision('high')
print(torch.cuda.get_device_name(0), torch.__version__)
REPO = '/content/cache_arch'
PUBLIC_URL = 'https://github.com/Angelawoo572/cache_arch.git'
TOKEN = userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add a private-repository GITHUB_TOKEN in Colab Secrets'
ASKPASS = '/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo x-access-token ;;\n  *) echo "$GITHUB_TOKEN" ;;\nesac\n')
os.chmod(ASKPASS, 0o700)
git_env = os.environ.copy()
git_env.update({'GIT_ASKPASS': ASKPASS, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': TOKEN})
try:
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', PUBLIC_URL, REPO], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', 'origin', 'main'], check=True, env=git_env)
finally:
    pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(subprocess.check_output(['git', '-C', REPO, 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ID = '602_offline_lstm_stride_stateful_v2_seed7'
DRIVE_ROOT = f'/content/drive/MyDrive/cache_prefetch_602/{RUN_ID}'
INPUT_DIR = f'{DRIVE_ROOT}/colab_input'
OUTPUT_ROOT = f'{DRIVE_ROOT}/colab_output'
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
ARCHIVE = f'{DRIVE_ROOT}/{RUN_ID}.colab_input.tar.gz'
if os.path.isfile(ARCHIVE):
    subprocess.run(['tar', '-xzf', ARCHIVE, '-C', INPUT_DIR], check=True)
print('Input archive:', ARCHIVE)
print('Extracted input:', INPUT_DIR)
print('Capacity-sweep outputs will persist in:', OUTPUT_ROOT)

In [ ]:
import json, pathlib
TRACE = '602.gcc_s-734B'
LOCAL_INPUT_DIR = f'/content/{RUN_ID}_colab_input'
LOCAL_OUTPUT_ROOT = f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_INPUT_DIR): shutil.rmtree(LOCAL_INPUT_DIR)
if os.path.isdir(LOCAL_OUTPUT_ROOT): shutil.rmtree(LOCAL_OUTPUT_ROOT)
shutil.copytree(INPUT_DIR, LOCAL_INPUT_DIR)
TRAIN = f'{LOCAL_INPUT_DIR}/{TRACE}.train_stream.csv.gz'
EVAL = f'{LOCAL_INPUT_DIR}/{TRACE}.eval_stream.csv.gz'
assert os.path.isfile(TRAIN), TRAIN
assert os.path.isfile(EVAL), EVAL
SCRIPT = f'{REPO}/formal_NN_training/experiments/602_offline_lstm_stride/python/train_and_offline_infer.py'
# Five fixed, predeclared widths: 545, 1,729, 6,017, 22,273, 85,505 parameters.
HIDDEN_SIZES = [8, 16, 32, 64, 128]
SWEEP = []
for hidden_size in HIDDEN_SIZES:
    tag = f'h{hidden_size}'
    out_dir = f'{LOCAL_OUTPUT_ROOT}/{tag}'
    drive_out_dir = f'{OUTPUT_ROOT}/{tag}'
    if os.path.isdir(out_dir): shutil.rmtree(out_dir)
    if os.path.isdir(drive_out_dir): shutil.rmtree(drive_out_dir)
    cmd = [sys.executable, SCRIPT, '--train-stream', TRAIN, '--eval-stream', EVAL,
           '--out-dir', out_dir, '--device', 'cuda', '--seed', '7', '--epochs', '8',
           '--chunk-len', '1024', '--batch-chunks', '64', '--hidden-size', str(hidden_size)]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
    metadata = json.loads(pathlib.Path(f'{out_dir}/run_metadata.json').read_text())
    assert metadata['training_state_mode'] == 'chronological_stateful_tbptt'
    assert metadata['training_chunks_shuffled'] is False
    assert metadata['training_state_carried_across_chunks'] is True
    assert metadata['experiment_revision'] == 'stateful_tbptt_v2'
    shutil.copytree(out_dir, drive_out_dir)
    SWEEP.append({'model_tag': tag, 'hidden_size': hidden_size,
                  'parameter_count': metadata['parameter_count'],
                  'threshold': metadata['threshold'],
                  'offline_lstm_entries': metadata['offline_lstm_entries']})
pathlib.Path(f'{LOCAL_OUTPUT_ROOT}/sweep_manifest.json').write_text(json.dumps({'trace': TRACE, 'experiment_revision': 'stateful_tbptt_v2', 'points': SWEEP}, indent=2) + '\n')
shutil.copy2(f'{LOCAL_OUTPUT_ROOT}/sweep_manifest.json', f'{OUTPUT_ROOT}/sweep_manifest.json')
print(json.dumps(SWEEP, indent=2))

In [ ]:
import json, pathlib
required = ['offline_stride.replay.csv', 'offline_lstm.replay.csv', 'model.pt', 'run_metadata.json']
for hidden_size in HIDDEN_SIZES:
    out_dir = f'{LOCAL_OUTPUT_ROOT}/h{hidden_size}'
    assert all(os.path.isfile(f'{out_dir}/{name}') for name in required), out_dir
print('Validated capacity points:', [f'h{x}' for x in HIDDEN_SIZES])
OUTPUT_ARCHIVE = f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
LOCAL_STAGE = f'/content/{RUN_ID}_colab_output_stage'
LOCAL_ARCHIVE = f'/content/{RUN_ID}.colab_output.tar.gz'
if os.path.isdir(LOCAL_STAGE):
    shutil.rmtree(LOCAL_STAGE)
shutil.copytree(LOCAL_OUTPUT_ROOT, LOCAL_STAGE)
with tarfile.open(LOCAL_ARCHIVE, 'w:gz') as archive:
    archive.add(LOCAL_STAGE, arcname='.')
shutil.copy2(LOCAL_ARCHIVE, OUTPUT_ARCHIVE)
print('DONE. Copy this archive back to Linux:', OUTPUT_ARCHIVE)